# Tarea 2 · Parte 1 — Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Podemos convertir documentos escaneados y entrevistas en audio peruanas en un archivo de texto estructurado y buscable?

| | |
|---|---|
| **Estudiante** | RoddyHuarhua (@roddyhuarhua-hub) |
| **Fecha de extracción** | se imprime en la celda siguiente (ejecución del notebook) |
| **Documentos (corpus A)** | 25 imágenes de Wikimedia Commons: 8 impresos, 8 fotos con sombra o inclinación, 9 manuscritos. Licencias libres (CC BY, CC BY-SA, CC0 o dominio público), registradas archivo por archivo en `datos/fuentes.csv`. 9 de ellos son los que usó la Sesión 10; el resto sale de búsquedas en Commons. |
| **Audio (corpus B)** | OpenSLR 73 *Crowdsourced high-quality Peruvian Spanish speech* (Google, **CC BY-SA 4.0**), 20 clips de ~80 s de un solo hablante + 1 entrevista de 2 hablantes; y 3 entrevistas reales de Wikimedia Commons (**CC BY-SA 4.0**: El Agustino, Lima ×2; Aurora Coronado, Fenmucarinap). Atribución en `datos/fuentes.csv`. |
| **OCR** | PaddleOCR 3.x, PP-OCRv5, `lang="es"`, motor PyTorch (`engine="transformers"`), sin corrección de orientación ni "planchado" de página, CPU. |
| **ASR** | faster-whisper `small` (CTranslate2, int8, CPU), `language="es"`, `beam_size=5`, **VAD activado** (Silero), diarización con huellas WeSpeaker + clustering (voz.py, Sesión 13). |
| **Entorno** | versiones exactas en la celda siguiente. |

Los archivos originales se guardan en `datos/raw/` y **no** se suben a Git (ver `.gitignore`); `datos/fuentes.csv` explica cómo recuperarlos (`python -c "import t2; t2.recolectar_documentos(); t2.recolectar_audio()"`).

In [ ]:
import sys, json, platform, datetime, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.express as px
from IPython.display import Image, display
import t2

pd.set_option("display.max_colwidth", 120)
FIGS = Path("figuras"); FIGS.mkdir(exist_ok=True)

def mostrar(fig, nombre):
    """Muestra el gráfico interactivo y además una copia PNG para que se vea en GitHub."""
    fig.write_html(FIGS / f"{nombre}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(FIGS / f"{nombre}.png", width=900, height=450, scale=1)
        display(Image(filename=str(FIGS / f"{nombre}.png")))
    except Exception as e:
        print("(sin PNG estático:", type(e).__name__, ")")
    fig.show()

import importlib.metadata as md
def ver(p):
    try: return md.version(p)
    except Exception: return "no instalado"
print("Fecha de extracción:", datetime.date.today().isoformat())
print("Python", platform.python_version(), "|", platform.platform())
for p in ["paddleocr", "paddlex", "torch", "transformers", "faster-whisper", "ctranslate2", "pyannote.audio", "easyocr", "jiwer", "plotly"]:
    print(f"  {p:15s} {ver(p)}")

## 1. Corpus A: documentos
Se descargan con la API de Commons (User-Agent descriptivo, 1 s de pausa entre pedidos), versión de 1600 px de ancho. Solo se aceptan licencias libres; los duplicados se detectan por hash SHA-1 del archivo.

In [ ]:
docs = t2.recolectar_documentos()
print("Documentos:", len(docs)); print(docs["tipo"].value_counts().to_string())
docs[["id", "tipo", "titulo_commons", "licencia"]]

In [ ]:
try:
    desc_doc = pd.read_csv("datos/raw/documentos/descartes_descarga.csv")
except Exception:
    desc_doc = pd.DataFrame(columns=["archivo", "motivo"])
print("Candidatos descartados al descargar:", len(desc_doc))
desc_doc

## 2. Corpus B: audio
OpenSLR 73 se lee por partes (HTTP Range) con `datos_sesion13.py` (copia de `datos.py` de la Sesión 13): no se bajan los 900 MB de cada zip. Cada clip une 16 frases seguidas de **un mismo hablante**, con 0,5 s de silencio entre frases.

In [ ]:
aud = t2.recolectar_audio()
print(f"Clips: {len(aud)} | minutos: {aud.duracion_seg.sum()/60:.1f} | multihablante: {(aud.n_hablantes_real == 2).sum()} + entrevistas reales")
aud[["id", "fuente", "condicion", "duracion_seg", "licencia"]]

## 3. Limpieza: duplicados por hash y archivos sin contenido útil

In [ ]:
import hashlib
dup_doc = docs[docs.duplicated("sha1")]
dup_aud = aud[aud.duplicated("sha1")]
print("Documentos duplicados por hash:", len(dup_doc), "| audios duplicados por hash:", len(dup_aud))

import voz
voz_seg = {}
for r in aud.itertuples():
    x = voz.cargar_audio(r.archivo_audio)
    voz_seg[r.id] = sum(b - a for a, b in voz.tramos_de_voz(x))
aud["segundos_con_voz"] = aud.id.map(voz_seg).round(1)
sin_voz = aud[aud.segundos_con_voz < 3]
print("Audios sin voz (VAD < 3 s de habla):", len(sin_voz))
aud = aud[~aud.id.isin(sin_voz.id) & ~aud.id.isin(dup_aud.id)].reset_index(drop=True)
aud[["id", "duracion_seg", "segundos_con_voz"]].T

## 4. OCR con PaddleOCR (los 25 documentos)
Respaldo crudo (texto, confianza, número de líneas y tiempo por página) en `datos/raw/documentos/ocr_paddle_base.json`.

In [ ]:
rutas_doc = [Path(a) for a in docs.archivo]
ocr = t2.ocr_paddle(rutas_doc)
docs = docs.merge(ocr.rename(columns={"archivo": "nombre"}), left_on=docs.archivo.str.split("/").str[-1], right_on="nombre").drop(columns=["key_0", "nombre"], errors="ignore")
docs["n_caracteres"] = docs.texto.str.len()
print(f"Tiempo total OCR: {docs.segundos.sum():.0f} s ({docs.segundos.mean():.1f} s por página)")
docs[["id", "tipo", "n_lineas", "confianza_media", "n_caracteres", "segundos"]].round(2)

In [ ]:
# Documentos ilegibles: menos de 20 caracteres reconocidos o confianza media < 0,5
ilegibles = docs[(docs.n_caracteres < 20) | (docs.confianza_media < 0.5)]
print("Documentos marcados como ilegibles:", len(ilegibles))
ilegibles[["id", "tipo", "titulo_commons", "n_caracteres", "confianza_media"]]

## 5. ASR con faster-whisper + VAD + diarización
Todos los clips con faster-whisper `small` en CPU y VAD. La diarización se aplica a los audios con más de un hablante: la entrevista de 2 hablantes de OpenSLR (de la que conocemos la verdad) y la entrevista callejera de El Agustino con dos vendedoras. Se exportan SRT y TXT por clip a `datos/raw/audio/salidas/`.

In [ ]:
multi = list(aud[(aud.n_hablantes_real == 2) | aud.condicion.str.contains("dos vendedoras")].id)
print("Se diarizan:", multi)
asr = t2.asr_faster_whisper([Path(a) for a in aud.archivo_audio], "small", diarizar_ids=tuple(multi))
aud = aud.merge(asr.drop(columns=["duracion_seg"]), left_on=aud.archivo_audio.str.split("/").str[-1], right_on="archivo").drop(columns=["key_0", "archivo"], errors="ignore")
aud["rtf"] = (aud.segundos_proceso / aud.duracion_seg).round(3)
aud["n_palabras"] = aud.texto.str.split().str.len()
print(f"Audio transcrito: {aud.duracion_seg.sum()/60:.1f} min en {aud.segundos_proceso.sum()/60:.1f} min de CPU (RTF medio {aud.rtf.mean():.2f})")
aud[["id", "duracion_seg", "n_palabras", "n_hablantes", "rtf"]]

In [ ]:
for i in multi:
    print(f"── {i}: diarización ({aud.set_index('id').n_hablantes[i]} hablantes detectados)")
    print(Path(f"datos/raw/audio/salidas/fw_small/{i}.txt").read_text(encoding="utf-8")[:700], "\n")

## 6. Seudonimización
Regla: se reemplazan DNI (8 dígitos), RUC, teléfonos celulares y correos, y los nombres de **personas privadas** que aparezcan (por ejemplo, vendedores entrevistados en la calle). Los nombres de figuras históricas o públicas que firman documentos públicos (Ricardo Palma, firmantes del Acta de Independencia) no son datos personales protegidos y se mantienen. La lista `NOMBRES_PRIVADOS` se completa después de **leer cada texto**: el método automático no reemplaza la revisión humana.

In [ ]:
# Completar tras la revisión manual de los textos: {"Nombre real": "Seudónimo"}
NOMBRES_PRIVADOS = {}
conteo = {}
for df, col in [(docs, "texto"), (aud, "texto")]:
    nuevos, ns = zip(*[t2.seudonimizar(t, NOMBRES_PRIVADOS) for t in df[col]])
    df[col] = list(nuevos); conteo[col + ("_ocr" if df is docs else "_asr")] = sum(ns)
    df["reemplazos"] = list(ns)
print("Entidades reemplazadas:", conteo)
print(pd.concat([docs.loc[docs.reemplazos > 0, ["id", "reemplazos"]], aud.loc[aud.reemplazos > 0, ["id", "reemplazos"]]]).to_string(index=False) or "ninguna")

## 7. Análisis exploratorio (Plotly Express)

In [ ]:
cuenta = docs.tipo.value_counts().rename_axis("tipo").reset_index(name="documentos")
ch = docs.groupby("tipo").n_caracteres.median()
titulo = f"Corpus balanceado (8/8/9), pero los {ch.idxmin()}s rinden menos texto: mediana de {ch.min():.0f} caracteres vs {ch.max():.0f} en {ch.idxmax()}s"
fig = px.bar(cuenta, x="tipo", y="documentos", text="documentos", title=titulo, color="tipo")
mostrar(fig, "g1_documentos_por_tipo")

In [ ]:
aud["fuente_corta"] = np.where(aud.fuente.str.startswith("OpenSLR"), "OpenSLR 73", "Wikimedia Commons")
m = aud.groupby("fuente_corta").duracion_seg.sum() / 60
titulo = f"{m.get('OpenSLR 73', 0):.0f} de {m.sum():.0f} minutos vienen de habla leída (OpenSLR); las entrevistas reales son pocos clips pero largos"
fig = px.bar(aud.sort_values("duracion_seg"), x="id", y="duracion_seg", color="fuente_corta", title=titulo,
             labels={"duracion_seg": "duración (s)", "id": "clip", "fuente_corta": "fuente"})
mostrar(fig, "g2_duracion_por_clip")

In [ ]:
largo = pd.concat([pd.DataFrame({"modalidad": "OCR (caracteres)", "longitud": docs.n_caracteres}),
                   pd.DataFrame({"modalidad": "ASR (palabras)", "longitud": aud.n_palabras})])
titulo = (f"El OCR es muy desigual ({docs.n_caracteres.min()}–{docs.n_caracteres.max()} caracteres); "
          f"el ASR es homogéneo salvo las entrevistas largas ({aud.n_palabras.median():.0f} palabras de mediana)")
fig = px.histogram(largo, x="longitud", facet_col="modalidad", nbins=20, title=titulo)
fig.update_xaxes(matches=None)
mostrar(fig, "g3_longitud_textos")

## 8. Validación: extremos de cada corpus
Comprobamos que los extremos de los histogramas tienen sentido mirando el texto.

In [ ]:
cols = ["id", "tipo", "n_caracteres", "confianza_media"]
print("OCR más largos"); display(docs.nlargest(3, "n_caracteres")[cols + ["texto"]])
print("OCR más cortos"); display(docs.nsmallest(3, "n_caracteres")[cols + ["texto"]])

In [ ]:
cols = ["id", "condicion", "duracion_seg", "n_palabras"]
print("Clips más largos"); display(aud.nlargest(3, "duracion_seg")[cols + ["texto"]])
print("Clips más cortos"); display(aud.nsmallest(3, "duracion_seg")[cols + ["texto"]])

## 9. Exportación a CSV

In [ ]:
documentos = pd.DataFrame({
    "id": docs.id, "archivo": docs.archivo, "tipo": docs.tipo,
    "fuente": "Wikimedia Commons (" + docs.licencia + ")",
    "texto_ocr": docs.texto,
    "modelo_ocr": "PaddleOCR " + ver("paddleocr") + " PP-OCRv5 (lang=es, engine=transformers, CPU, sin orientación/unwarping)",
    "texto_referencia": "",
})
transcripciones = pd.DataFrame({
    "id": aud.id, "archivo_audio": aud.archivo_audio, "duracion_seg": aud.duracion_seg,
    "idioma": "es (variante peruana documentada por la fuente)", "texto_asr": aud.texto,
    "modelo_asr": "faster-whisper " + ver("faster-whisper") + " small (int8, CPU, VAD on, beam 5)",
    "n_hablantes": aud.n_hablantes.fillna(1).astype(int), "texto_referencia": "",
})
fuentes = pd.concat([
    pd.DataFrame({"id": docs.id, "modalidad": "documento", "fuente_url": docs.fuente_url, "licencia": docs.licencia,
                  "autor": docs.autor, "acceso": "API de Commons; t2.recolectar_documentos() (ancho 1600 px)"}),
    pd.DataFrame({"id": aud.id, "modalidad": "audio", "fuente_url": aud.fuente_url, "licencia": aud.licencia,
                  "autor": aud.fuente, "acceso": "t2.recolectar_audio() (OpenSLR por HTTP Range; Commons Special:FilePath)"}),
])
documentos.to_csv("datos/documentos.csv", index=False)
transcripciones.to_csv("datos/transcripciones.csv", index=False)
fuentes.to_csv("datos/fuentes.csv", index=False)
aud[["id", "referencia_dataset", "condicion", "fuente_corta", "rtf", "segundos_proceso"]].to_csv("datos/raw/audio/meta_clips.csv", index=False)
for nombre, df in [("documentos", documentos), ("transcripciones", transcripciones), ("fuentes", fuentes)]:
    print(nombre, df.shape)
documentos.head()

In [ ]:
transcripciones.head()

In [ ]:
fuentes.head()

In [ ]:
# Ejemplos publicables para el dashboard (licencia libre, sin datos personales): todas las imágenes reducidas y 6 clips cortos
from PIL import Image as PImg
import soundfile as sf
(t2.EJEMPLOS / "img").mkdir(exist_ok=True); (t2.EJEMPLOS / "audio").mkdir(exist_ok=True)
for r in docs.itertuples():
    im = PImg.open(r.archivo).convert("RGB"); im.thumbnail((900, 900)); im.save(t2.EJEMPLOS / "img" / f"{r.id}.jpg", quality=80)
for i in aud.id[:3].tolist() + aud.id[10:13].tolist() + multi:
    x, sr = sf.read(f"datos/raw/audio/{i}.wav"); sf.write(t2.EJEMPLOS / "audio" / f"{i}.ogg", x[: 45 * sr], sr, format="OGG", subtype="VORBIS")
print(sum(p.stat().st_size for p in t2.EJEMPLOS.rglob("*") if p.is_file()) / 1e6, "MB en datos/ejemplos/")